In [1]:
import os
import torch
import torch.nn as nn
import torch.utils.data as Data
import torchvision
import torch.nn.functional as F
import numpy as np

learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 3
BATCH_SIZE = 50

# 如果本地没有mnist数据就自动下载
DOWNLOAD_MNIST = False
if not(os.path.exists('./mnist/')) or not os.listdir('./mnist/'):
    DOWNLOAD_MNIST = True

# 加载训练集
train_data = torchvision.datasets.MNIST(
    root='./mnist/',
    train=True,
    transform=torchvision.transforms.ToTensor(),
    download=DOWNLOAD_MNIST,
)
train_loader = Data.DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# 加载测试集（取前500张）
# 注意：新版PyTorch用 .data 和 .targets 代替旧版的 .test_data 和 .test_labels
test_data = torchvision.datasets.MNIST(root='./mnist/', train=False)
test_x = torch.unsqueeze(test_data.data, dim=1).type(torch.FloatTensor)[:500] / 255.
test_y = test_data.targets[:500].numpy()


class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # ===== 第一层卷积 =====
        # 输入：1通道（灰度图），输出：32通道
        # 卷积核大小：7×7，步长：1
        # padding=3 是为了保持卷积前后尺寸不变（same padding）
        #   公式：padding = (kernel_size - 1) // 2 = (7-1)//2 = 3
        # 28×28 → (经过卷积，same padding) → 28×28 → (MaxPool2d(2)) → 14×14
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                in_channels=1,    # 输入通道数：MNIST是灰度图，所以是1
                out_channels=32,  # 输出通道数：32个卷积核
                kernel_size=7,    # 卷积核大小：7×7
                stride=1,         # 步长：1
                padding=3,        # same padding：(7-1)//2 = 3
            ),
            nn.ReLU(),            # 激活函数
            nn.MaxPool2d(2),      # 最大池化，尺寸减半：14×14
        )

        # ===== 第二层卷积 =====
        # 输入：32通道（来自conv1输出），输出：64通道
        # 卷积核大小：5×5，步长：1
        # padding=2 保持尺寸不变（same padding）
        #   公式：padding = (5-1)//2 = 2
        # 14×14 → (经过卷积，same padding) → 14×14 → (MaxPool2d(2)) → 7×7
        self.conv2 = nn.Sequential(
            nn.Conv2d(
                in_channels=32,   # 输入通道：来自conv1的输出
                out_channels=64,  # 输出通道：64个卷积核
                kernel_size=5,    # 卷积核大小：5×5
                stride=1,         # 步长：1
                padding=2,        # same padding：(5-1)//2 = 2
            ),
            nn.ReLU(),            # 激活函数
            nn.MaxPool2d(2),      # 最大池化，尺寸减半：7×7
        )

        self.out1 = nn.Linear(7 * 7 * 64, 1024, bias=True)  # 全连接层1
        self.dropout = nn.Dropout(keep_prob_rate)             # Dropout防止过拟合
        self.out2 = nn.Linear(1024, 10, bias=True)            # 全连接层2，输出10个类别

    def forward(self, x):
        x = self.conv1(x)              # 输出形状：(batch, 32, 14, 14)
        x = self.conv2(x)              # 输出形状：(batch, 64, 7, 7)

        # 把三维特征图展平成一维向量
        # -1 表示自动计算batch_size，7*7*64=3136 是每张图的特征数
        x = x.view(-1, 7 * 7 * 64)    # 输出形状：(batch, 3136)

        out1 = self.out1(x)            # 全连接层1：3136 → 1024
        out1 = F.relu(out1)            # ReLU激活
        out1 = self.dropout(out1)      # Dropout
        out2 = self.out2(out1)         # 全连接层2：1024 → 10
        output = F.softmax(out2, dim=1)  # softmax转为概率（加dim=1避免警告）
        return output


def test(cnn):
    """在测试集上计算准确率"""
    global prediction
    cnn.eval()  # 切换到评估模式（关闭Dropout）
    with torch.no_grad():
        y_pre = cnn(test_x)
    _, pre_index = torch.max(y_pre, 1)
    pre_index = pre_index.view(-1)
    prediction = pre_index.data.numpy()
    correct = np.sum(prediction == test_y)
    cnn.train()  # 切换回训练模式
    return correct / 500.0


def train(cnn):
    """训练模型"""
    optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate)
    loss_func = nn.CrossEntropyLoss()  # 交叉熵损失

    for epoch in range(max_epoch):
        print(f'\n===== Epoch {epoch+1}/{max_epoch} =====')
        for step, (x_, y_) in enumerate(train_loader):
            output = cnn(x_)
            loss = loss_func(output, y_)
            optimizer.zero_grad()   # 清空梯度
            loss.backward()         # 反向传播
            optimizer.step()        # 更新参数

            if step != 0 and step % 20 == 0:
                acc = test(cnn)
                print(f'Step {step:4d} | Loss: {loss.item():.4f} | Test Accuracy: {acc*100:.1f}%')


if __name__ == '__main__':
    cnn = CNN()
    print('模型结构：')
    print(cnn)
    train(cnn)
    print('\n最终测试精度：', test(cnn))


Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9912422/9912422 [00:11<00:00, 835535.15it/s] 


Extracting ./mnist/MNIST\raw\train-images-idx3-ubyte.gz to ./mnist/MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28881/28881 [00:00<00:00, 193436.74it/s]


Extracting ./mnist/MNIST\raw\train-labels-idx1-ubyte.gz to ./mnist/MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1648877/1648877 [00:00<00:00, 1692319.74it/s]


Extracting ./mnist/MNIST\raw\t10k-images-idx3-ubyte.gz to ./mnist/MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4542/4542 [00:00<00:00, 933574.87it/s]


Extracting ./mnist/MNIST\raw\t10k-labels-idx1-ubyte.gz to ./mnist/MNIST\raw

模型结构：
CNN(
  (conv1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (out1): Linear(in_features=3136, out_features=1024, bias=True)
  (dropout): Dropout(p=0.7, inplace=False)
  (out2): Linear(in_features=1024, out_features=10, bias=True)
)

===== Epoch 1/3 =====
Step   20 | Loss: 2.2855 | Test Accuracy: 30.8%
Step   40 | Loss: 2.2290 | Test Accuracy: 31.2%
Step   60 | Loss: 2.0997 | Test Accuracy: 45.6%
Step   80 | Loss: 1.9659 | Test Accuracy: 60.4%
Step  100 | Loss: 1.9745 | Test Accuracy: 67.4%
Step  120 | Loss: 1.7506 | Test Accuracy: 70.4%
Step  140 | Loss: 1.7477 